In [3]:
from langchain_core.documents import Document 
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter # uv add langchain_text_splitters
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.callbacks import BaseCallbackHandler
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_classic.retrievers import ParentDocumentRetriever, MultiVectorRetriever  # uv add langchain_classic , used for small2big retrievers
from langchain_core.stores import InMemoryStore
from pydantic import BaseModel

In [4]:
class CallCounter(BaseCallbackHandler):
    def __init__(self):
        self.count = 0

    def on_chat_model_start(self, *args, **kwargs):
        self.count += 1

CALLS = CallCounter()

def make_chat_model(model_name="gpt-4o-mini"):
    return ChatOpenAI(
        model=model_name,
        callbacks=[CALLS]
    )

model = make_chat_model()
    

In [5]:
def to_document(doc, dept, status, text):
    return Document(page_content=text, metadata={
        "doc": doc,
        "dept": dept,
        "status": status
    })


CORPUS = [
    to_document("refund_current", "policy", "current", "Refund policy. Annual subscriptions can be refunded within 14 days of purchase. After 14 days we offer store credit for the unused months. Monthly subscriptions can be cancelled at any time, but the current month is not refunded."),
    to_document("faq_refund", "faq", "current", "FAQ: How do refunds work? You can get a refund on an annual subscription within 14 days of buying it. Monthly plans are never refunded for the month already started, but you can cancel any time."),
    to_document("refund_2023", "policy", "archived", "ARCHIVED (2023) refund policy: annual subscriptions could be refunded within 30 days of purchase."),
    to_document("shipping", "policy", "current", "Shipping. Orders of 5 bags or more ship free. Orders placed before 12:00 UTC ship the same day. Standard delivery takes 3 to 5 working days in the UK and 7 to 10 working days elsewhere in Europe."),
    to_document("roasting", "ops", "current", "Roasting schedule. Light roasts are roasted every Tuesday and dark roasts every Friday at our Lisbon roastery. Beans are freshest for 30 days after the roast date when kept in a sealed bag."),
    to_document("blends", "product", "current", "The Drift blend is a light roast from Ethiopia with notes of jasmine and lemon. The Ember blend is a dark roast with notes of cocoa and orange peel."),
    to_document("support", "policy", "current", "Customer support is open 9:00 to 17:00 UTC, Monday to Friday. Premium subscribers get email replies within 4 hours."),
    to_document("giftcards", "policy", "current", "Zorvia gift cards never expire and can be used on subscriptions and single bags."),
    to_document("wholesale", "sales", "current", "Wholesale accounts require a minimum order of 20 kg per month and get 15% off list price."),
    to_document("referral", "marketing", "current", "Referral programme: both the referrer and the friend get one free bag after the friend's first order."),
    to_document("equipment", "product", "current", "Equipment note: the GX-4471 hand grinder is out of stock until March. The GX-5000 electric grinder ships in 2 days."),
    to_document("brewing", "product", "current", "Brewing guide for the Drift blend. Use 15 g of coffee for 250 ml of water at 94 degrees C. Wet the grounds and wait 30 seconds, then pour slowly in circles. Total brew time should be about 3 minutes. If the cup tastes sour, grind finer; if it tastes bitter, grind coarser. Rinse the paper filter first to remove any papery taste."),
    to_document("company", "about", "current", "Zorvia Coffee Co. was founded in Lisbon in 2018 and ships compostable packaging."),
]

print(len(CORPUS), "documents")

13 documents


In [6]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True}
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

/Users/abhishek/dev/learning/agentic-ai/ai-engineer-cohort/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1025 16:51:06.824000 3496 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13907.20it/s]


In [7]:
CURRENT = lambda d: d.metadata.get("status") == "current"

def build_all(docs=None):
    global store, chunks
    chunks = splitter.split_documents(docs or CORPUS)
    store = InMemoryVectorStore(embedding=embedding)
    store.add_documents(chunks)

In [8]:
build_all()

print(len(chunks), "chunks")

14 chunks


In [9]:
vec = embedding.embed_query("What is the refund policy?")
print(len(vec), vec[:6])

384 [-0.09238819777965546, 0.05380009487271309, 0.0462050586938858, 0.029378825798630714, 0.04612403362989426, 0.046625617891550064]


In [10]:
def retrieve(question, k=3, filter=None):
    kwargs = {"filter": filter} if filter else {}
    return store.similarity_search_with_score(question, k=k, **kwargs)

def show(results):
    for doc, score in results:
        print(f"{score:.3f} [{doc.metadata['status']}] {doc.metadata['doc']}: {doc.page_content[:100]}....")

In [11]:
QUESTION = "What is the refund policy?"
show(retrieve(QUESTION, k=3))

0.610 [archived] refund_2023: ARCHIVED (2023) refund policy: annual subscriptions could be refunded within 30 days of purchase.....
0.594 [current] faq_refund: FAQ: How do refunds work? You can get a refund on an annual subscription within 14 days of buying it....
0.570 [current] refund_current: Refund policy. Annual subscriptions can be refunded within 14 days of purchase. After 14 days we off....


In [12]:
QUESTION = "What is the refund policy?"
show(retrieve(QUESTION, k=3, filter=CURRENT))

0.594 [current] faq_refund: FAQ: How do refunds work? You can get a refund on an annual subscription within 14 days of buying it....
0.570 [current] refund_current: Refund policy. Annual subscriptions can be refunded within 14 days of purchase. After 14 days we off....
0.272 [current] referral: Referral programme: both the referrer and the friend get one free bag after the friend's first order....


In [13]:
SYSTEM_PROMPT = """
You are a helpful assistant for Zorvia Coffee Co. Use only the context below. If the context doesnot contain the answer, say "I don't know".
Keep your answers concise, short and to the point.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT + "\n\n" + "Context:\n{context}"),
    ("human", "{question}"),
])

answer_chain = prompt | model | StrOutputParser()

def format_docs(docs):
    return "\n".join(f"- {d.page_content}" for d in docs)

def make_chain(retriever):
    return {"context": retriever | format_docs, "question": RunnablePassthrough() } | answer_chain

def naive_rag(question, k=3, filter=None):
    retriever = store.as_retriever(
        search_kwargs={"k": k, **({"filter": filter} if filter else {})}
    )
    return make_chain(retriever).invoke(question)

In [14]:
naive_rag(QUESTION)

'Annual subscriptions can be refunded within 14 days of purchase. After 14 days, store credit is offered for unused months. Monthly subscriptions can be canceled at any time, but the current month is not refunded.'

# Small 2 Big Retriever optimisation

In [15]:
small_to_big = ParentDocumentRetriever(
    vectorstore=InMemoryVectorStore(embedding=embedding),
    docstore=InMemoryStore(),
    child_splitter=RecursiveCharacterTextSplitter(chunk_size=75, chunk_overlap=10),
    parent_splitter=RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50),
    search_kwargs={"k": 3, **({"filter": CURRENT} if CURRENT else {})},
)

In [16]:
small_to_big.add_documents(CORPUS)

In [17]:
QUESTION_FOR_SMALL2BIG = "How fast do premium subscribers get support or expect a reply?"


print("Parent: ", small_to_big.invoke(QUESTION_FOR_SMALL2BIG)[0].page_content)
print("Child: ", small_to_big.vectorstore.similarity_search(QUESTION_FOR_SMALL2BIG, k=1))

Parent:  Customer support is open 9:00 to 17:00 UTC, Monday to Friday. Premium subscribers get email replies within 4 hours.
Child:  [Document(id='6ec81faa-90f7-4238-b6c5-14a10457a250', metadata={'doc': 'support', 'dept': 'policy', 'status': 'current', 'doc_id': '39a57bbb-fb32-456e-92a9-a69924623192'}, page_content='Premium subscribers get email replies within 4 hours.')]


In [18]:
def naive_rag_with_small2big(question):
    return make_chain(small_to_big).invoke(question)

naive_rag_with_small2big(QUESTION_FOR_SMALL2BIG)

'Premium subscribers get email replies within 4 hours.'

# Reverse HyDE

In [19]:
class QuestionsForHyDE(BaseModel):
    questions: list[str]

question_creator_chain = (
    ChatPromptTemplate.from_template("Write 3 different questions a customer might ask that this text answers.\nText: {text}")
    | model.with_structured_output(QuestionsForHyDE)
)

In [20]:
current_docs = [c for c in chunks if CURRENT(c)]

questions = question_creator_chain.batch([{"text": c.page_content} for c in current_docs])

In [21]:
questions

[QuestionsForHyDE(questions=['What is the refund policy for annual subscriptions?', 'Can I cancel my monthly subscription and get a refund for the current month?', 'What happens if I cancel my annual subscription after 14 days?']),
 QuestionsForHyDE(questions=['What is the refund policy for annual subscriptions?', "Can I get a refund if I'm on a monthly plan?", 'How long do I have to request a refund after purchasing an annual subscription?']),
 QuestionsForHyDE(questions=['Do I get free shipping if I order 5 bags or more?', 'What is the shipping time for orders placed before 12:00 UTC?', 'How long does standard delivery take for orders shipped outside the UK?']),
 QuestionsForHyDE(questions=['When are your light and dark roasts prepared?', 'How long are the coffee beans fresh after roasting?', 'What is the best way to store my coffee beans to ensure freshness?']),
 QuestionsForHyDE(questions=['What are the flavor notes of the Drift blend?', 'Can you tell me about the roast level of th

In [22]:
chunk_ids = [str(i)for i in range(len(current_docs))]

question_docs = [Document(page_content=qtext, metadata={"chunk_id": chunk_ids[i]}) for i, j in enumerate(questions) for qtext in j.questions]


In [23]:
question_docs

[Document(metadata={'chunk_id': '0'}, page_content='What is the refund policy for annual subscriptions?'),
 Document(metadata={'chunk_id': '0'}, page_content='Can I cancel my monthly subscription and get a refund for the current month?'),
 Document(metadata={'chunk_id': '0'}, page_content='What happens if I cancel my annual subscription after 14 days?'),
 Document(metadata={'chunk_id': '1'}, page_content='What is the refund policy for annual subscriptions?'),
 Document(metadata={'chunk_id': '1'}, page_content="Can I get a refund if I'm on a monthly plan?"),
 Document(metadata={'chunk_id': '1'}, page_content='How long do I have to request a refund after purchasing an annual subscription?'),
 Document(metadata={'chunk_id': '2'}, page_content='Do I get free shipping if I order 5 bags or more?'),
 Document(metadata={'chunk_id': '2'}, page_content='What is the shipping time for orders placed before 12:00 UTC?'),
 Document(metadata={'chunk_id': '2'}, page_content='How long does standard deli

In [25]:
reverse_hyde = MultiVectorRetriever(
    vectorstore=InMemoryVectorStore(embedding=embedding),
    docstore=InMemoryStore(),
    id_key="chunk_id", # this property controls vector store key to map multiple questions to a same chunk
    search_kwargs={"k": 2}
    
)

reverse_hyde.vectorstore.add_documents(question_docs)
reverse_hyde.docstore.mset(list(zip(chunk_ids, current_docs)))

QUESTION_FOR_REVERSE_HYDE = "How fast do premium subscribers get support or expect a reply?"

print("Parent: ", reverse_hyde.invoke(QUESTION_FOR_REVERSE_HYDE))

Parent:  [Document(metadata={'doc': 'support', 'dept': 'policy', 'status': 'current'}, page_content='Customer support is open 9:00 to 17:00 UTC, Monday to Friday. Premium subscribers get email replies within 4 hours.'), Document(metadata={'doc': 'faq_refund', 'dept': 'faq', 'status': 'current'}, page_content='FAQ: How do refunds work? You can get a refund on an annual subscription within 14 days of buying it. Monthly plans are never refunded for the month already started, but you can cancel any time.')]


In [26]:
def naive_rag_with_reverse_hyde(question):
    # reverse_hyde searches generated questions, then returns their source chunks.
    return make_chain(reverse_hyde).invoke(question)

naive_rag_with_reverse_hyde(QUESTION_FOR_REVERSE_HYDE)

'Premium subscribers get email replies within 4 hours.'